# Where interference estimates remain credible

## How Degree and Assignment Probability Shape Positivity

For the rule “at least one assigned neighbor,” a node of degree $d$ has

$$
\pi(0)=(1-p)^d,\qquad \pi(1)=1-(1-p)^d.
$$

The largest inverse probability is

$$
W_{\max}(d,p)=\max\{1/\pi(0),1/\pi(1)\}.
$$

Here, $p$ is the independent assignment probability. Large $W_{\max}$ means that at least one contrasted exposure is rare, so an observed unit in that exposure receives a large contribution.

In [1]:
from pathlib import Path
import io
import itertools
import math
import sys
import zipfile
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, display as _display

def display(value):
    module = value.__class__.__module__
    if module.startswith('pandas') and hasattr(value, '_repr_html_'):
        html = value._repr_html_()
        return _display(HTML('<div style="display:flex;justify-content:center;">' + html + '</div>'))
    return _display(value)
BOOK_ROOT = Path.cwd().parents[1]
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))
from book_support import ensure_dataset
FINANCIAL_ROOT = BOOK_ROOT
import RiskLabAI
from RiskLabAI.causal_factor_analysis import exposure_mean_contrast
assert RiskLabAI.__version__ == '3.2.0'
FIGURES = BOOK_ROOT / 'outputs' / 'figures' / '07_network_interference'
TABLES = BOOK_ROOT / 'outputs' / 'tables' / '07_network_interference'
FIGURE_DATA = BOOK_ROOT / 'outputs' / 'figure_data' / '07_network_interference'
DATA_FILE = ensure_dataset(BOOK_ROOT, 'french_49_industries')
for directory in [FIGURES, TABLES, FIGURE_DATA]:
    directory.mkdir(parents=True, exist_ok=True)
PALETTE = {'background': '#111827', 'axes': '#111827', 'text': '#E5E7EB', 'cyan': '#38BDF8', 'amber': '#FBBF24', 'green': '#34D399', 'rose': '#FB7185', 'violet': '#C084FC', 'grid': '#374151'}
plt.rcParams.update({'font.family': 'DejaVu Serif', 'font.size': 11.5, 'figure.facecolor': PALETTE['background'], 'axes.facecolor': PALETTE['axes'], 'axes.edgecolor': PALETTE['text'], 'axes.labelcolor': PALETTE['text'], 'axes.titlecolor': PALETTE['text'], 'xtick.color': PALETTE['text'], 'ytick.color': PALETTE['text'], 'text.color': PALETTE['text'], 'grid.color': PALETTE['grid'], 'grid.linestyle': ':', 'legend.facecolor': PALETTE['axes'], 'legend.edgecolor': PALETTE['grid'], 'savefig.facecolor': PALETTE['background'], 'savefig.transparent': False, 'savefig.dpi': 300})

def rendered_value(value):
    if pd.isna(value):
        return ''
    if isinstance(value, (bool, np.bool_)):
        return 'Pass' if bool(value) else 'Fail'
    if isinstance(value, (int, np.integer)):
        return f'{int(value):,}'
    if isinstance(value, (float, np.floating)):
        value = float(value)
        if value == 0:
            return '0'
        if value.is_integer():
            return f'{int(value):,}'
        if abs(value) < 0.0001 or abs(value) >= 1000000:
            return f'{value:.6g}'
        return f'{value:.6f}'.rstrip('0').rstrip('.')
    return str(value)

def rendered_frame(frame):
    return frame.apply(lambda column: column.map(rendered_value))

def save_table(frame, stem, caption, label, fit=False):
    shown = rendered_frame(frame)
    shown.to_csv(TABLES / f'{stem}.csv', index=False)
    latex = shown.to_latex(index=False, escape=True, caption=caption, label=label, position='htbp')
    latex = latex.replace('\\begin{table}[htbp]', '\\begin{table}[htbp]\n\\centering', 1)
    if fit:
        latex = latex.replace('\\begin{tabular}', '\\begin{adjustbox}{max width=\\textwidth}\\begin{tabular}', 1).replace('\\end{tabular}', '\\end{tabular}\\end{adjustbox}', 1)
    (TABLES / f'{stem}.tex').write_text(latex, encoding='utf-8')
    display(shown)
    return shown

def finish_axis(axis, grid=True):
    axis.spines['top'].set_visible(False)
    axis.spines['right'].set_visible(False)
    if grid:
        axis.grid(True, alpha=0.65)
    axis.set_axisbelow(True)

def save_figure(figure, stem):
    figure.savefig(FIGURES / f'{stem}.png', dpi=300, bbox_inches='tight', facecolor=PALETTE['background'], transparent=False)
    figure.savefig(FIGURES / f'{stem}.pdf', bbox_inches='tight', facecolor=PALETTE['background'], transparent=False)
    display(figure)
    plt.close(figure)

def read_public_returns():
    with zipfile.ZipFile(DATA_FILE) as archive:
        lines = archive.read(archive.namelist()[0]).decode('latin1').splitlines()
    header = next((index for index, line in enumerate(lines) if line.startswith(',Agric')))
    ending = next((index for index in range(header + 1, len(lines)) if not lines[index].strip()))
    raw = pd.read_csv(io.StringIO('\n'.join(lines[header:ending])))
    raw = raw.rename(columns={raw.columns[0]: 'Date'})
    raw['Date'] = pd.to_datetime(raw['Date'].astype(str), format='%Y%m%d')
    raw = raw.set_index('Date').replace([-99.99, -999.0], np.nan) / 100
    return raw.loc['2019-01-01':'2023-12-31'].dropna().copy()

def correlation_graph(correlation, neighbors):
    values = correlation.to_numpy()
    size = len(correlation)
    adjacency = np.zeros((size, size), dtype=int)
    for row in range(size):
        order = np.argsort(values[row])
        selected = [column for column in order[::-1] if column != row][:neighbors]
        adjacency[row, selected] = 1
    return np.maximum(adjacency, adjacency.T)

def line_graph(size):
    adjacency = np.zeros((size, size), dtype=int)
    for index in range(size - 1):
        adjacency[index, index + 1] = adjacency[index + 1, index] = 1
    return adjacency

def ring_graph(size, radius=1):
    adjacency = np.zeros((size, size), dtype=int)
    for index in range(size):
        for step in range(1, radius + 1):
            adjacency[index, (index + step) % size] = 1
            adjacency[index, (index - step) % size] = 1
    return adjacency

def star_graph(size):
    adjacency = np.zeros((size, size), dtype=int)
    adjacency[0, 1:] = 1
    adjacency[1:, 0] = 1
    return adjacency

def complete_graph(size):
    return np.ones((size, size), dtype=int) - np.eye(size, dtype=int)

def exposure_labels(adjacency, assignment, threshold=1):
    return (adjacency @ np.asarray(assignment) >= threshold).astype(int)

def exposure_probabilities(adjacency, assignment_probability, threshold=1):
    degrees = adjacency.sum(axis=1).astype(int)
    positive = np.array([1 - sum((math.comb(int(degree), count) * assignment_probability ** count * (1 - assignment_probability) ** (degree - count) for count in range(min(threshold, int(degree) + 1)))) for degree in degrees])
    return np.column_stack([1 - positive, positive])

def assignment_probability(assignment, probability):
    treated = int(np.asarray(assignment).sum())
    return probability ** treated * (1 - probability) ** (len(assignment) - treated)

def circle_positions(size):
    angles = np.linspace(np.pi / 2, np.pi / 2 + 2 * np.pi, size, endpoint=False)
    return np.column_stack([np.cos(angles), np.sin(angles)])

def draw_network(axis, adjacency, labels, node_values=None, title=''):
    positions = circle_positions(len(labels))
    for first in range(len(labels)):
        for second in range(first + 1, len(labels)):
            if adjacency[first, second]:
                axis.plot(positions[[first, second], 0], positions[[first, second], 1], color=PALETTE['grid'], linewidth=1.5, alpha=0.8, zorder=1)
    values = adjacency.sum(axis=1) if node_values is None else np.asarray(node_values)
    colors = [PALETTE['cyan'] if value <= np.median(values) else PALETTE['amber'] for value in values]
    axis.scatter(positions[:, 0], positions[:, 1], s=420, c=colors, edgecolors=PALETTE['text'], linewidths=1.0, zorder=2)
    for index, label in enumerate(labels):
        axis.text(positions[index, 0], positions[index, 1], str(label), ha='center', va='center', color=PALETTE['background'], fontsize=8.5, fontweight='bold', zorder=3)
    axis.set_title(title)
    axis.set_aspect('equal')
    axis.axis('off')
    return positions

def capture_failure(name, calculation):
    try:
        calculation()
        return {'Check': name, 'Rejected': False, 'Explanation': 'Calculation continued'}
    except ValueError:
        return {'Check': name, 'Rejected': True, 'Explanation': 'Stopped as required'}
degrees_grid = np.array([2, 4, 8, 12])
probabilities_grid = np.array([0.05, 0.1, 0.25, 0.5])
grid_rows = []
for degree in degrees_grid:
    for probability in probabilities_grid:
        probability_zero = (1 - probability) ** degree
        probability_one = 1 - probability_zero
        grid_rows.append({'Degree': degree, 'Assignment probability': probability, 'Probability of exposure zero': probability_zero, 'Probability of exposure one': probability_one, 'Largest inverse probability': max(1 / probability_zero, 1 / probability_one)})
probability_grid = pd.DataFrame(grid_rows)
save_table(probability_grid, 'scope_probability_grid', 'Exposure probabilities across degree and assignment probability.', 'tab:cim7-scope-probability-grid', fit=True)
selected_grid = probability_grid[probability_grid['Degree'].isin([2, 8, 12]) & probability_grid['Assignment probability'].isin([0.05, 0.25, 0.5])].reset_index(drop=True)
save_table(selected_grid, 'scope_probability_grid_selected', 'Selected positivity cases for the neighbor-exposure rule.', 'tab:cim7-scope-probability-grid-selected', fit=True)
probability_grid.to_csv(FIGURE_DATA / 'scope_probability_heatmap.csv', index=False)
matrix_probability = probability_grid.pivot(index='Degree', columns='Assignment probability', values='Probability of exposure zero')
figure, axis = plt.subplots(figsize=(8.4, 5.6), constrained_layout=True)
image = axis.imshow(matrix_probability, aspect='auto', vmin=0, vmax=1, cmap='viridis')
axis.set_xticks(np.arange(len(matrix_probability.columns)), [f'{value:.2f}' for value in matrix_probability.columns])
axis.set_yticks(np.arange(len(matrix_probability.index)), matrix_probability.index)
axis.set_xlabel('Assignment probability')
axis.set_ylabel('Degree')
axis.set_title('Probability of No Assigned Neighbor')
figure.colorbar(image, ax=axis, label='Exposure-zero probability')
save_figure(figure, 'scope_probability_heatmap')
probability_grid.to_csv(FIGURE_DATA / 'scope_inverse_weight_heatmap.csv', index=False)
matrix_weight = probability_grid.pivot(index='Degree', columns='Assignment probability', values='Largest inverse probability')
figure, axis = plt.subplots(figsize=(8.4, 5.6), constrained_layout=True)
image = axis.imshow(np.log10(matrix_weight), aspect='auto', cmap='magma')
axis.set_xticks(np.arange(len(matrix_weight.columns)), [f'{value:.2f}' for value in matrix_weight.columns])
axis.set_yticks(np.arange(len(matrix_weight.index)), matrix_weight.index)
axis.set_xlabel('Assignment probability')
axis.set_ylabel('Degree')
axis.set_title('Largest Inverse Exposure Probability')
colorbar = figure.colorbar(image, ax=axis)
colorbar.set_label('Base-10 logarithm of largest weight')
save_figure(figure, 'scope_inverse_weight_heatmap')


,Degree,Assignment probability,Probability of exposure zero,Probability of exposure one,Largest inverse probability
0,2,0.05,0.9025,0.0975,10.25641
1,2,0.1,0.81,0.19,5.263158
2,2,0.25,0.5625,0.4375,2.285714
3,2,0.5,0.25,0.75,4
4,4,0.05,0.814506,0.185494,5.391017
5,4,0.1,0.6561,0.3439,2.907822
6,4,0.25,0.316406,0.683594,3.160494
7,4,0.5,0.0625,0.9375,16
8,8,0.05,0.66342,0.33658,2.971066
9,8,0.1,0.430467,0.569533,2.323057


,Degree,Assignment probability,Probability of exposure zero,Probability of exposure one,Largest inverse probability
0,2,0.05,0.9025,0.0975,10.25641
1,2,0.25,0.5625,0.4375,2.285714
2,2,0.5,0.25,0.75,4
3,8,0.05,0.66342,0.33658,2.971066
4,8,0.25,0.100113,0.899887,9.988721
5,8,0.5,0.003906,0.996094,256
6,12,0.05,0.54036,0.45964,2.175616
7,12,0.25,0.031676,0.968324,31.569292
8,12,0.5,0.000244,0.999756,"4,096"


<Figure size 840x560 with 2 Axes>

<Figure size 840x560 with 2 Axes>

## Finite-Sample Variation Under Different Exposure Rules

For a threshold $r$, define exposure by

$$
D_i^{(r)}(Z)=\mathbf 1\!\left\{\sum_j A_{ij}Z_j\geq r\right\}.
$$

The case $r=1$ requires one assigned neighbor. The case $r=2$ requires at least two. Raising the threshold can make exposure one rare even when every marginal probability remains positive. The resulting increase in finite-sample variation is a property of the declared design and mapping.

In [2]:
simulation_random_state = 7301
simulation_repetitions = 2500
simulation_rows = []
for radius in [1, 2, 4, 6]:
    graph = ring_graph(40, radius)
    degree = int(graph[0].sum())
    baseline = 0.5 + 0.2 * np.sin(np.arange(40) / 3)
    effects = 0.6 + 0.1 * np.cos(np.arange(40) / 5)
    truth = float(effects.mean())
    for probability in probabilities_grid:
        exposure_probability = exposure_probabilities(graph, probability)
        rng = np.random.default_rng(simulation_random_state + 100 * radius + int(probability * 100))
        estimates = []
        for repetition in range(simulation_repetitions):
            assignment = rng.binomial(1, probability, 40)
            exposure = exposure_labels(graph, assignment)
            outcome = baseline + effects * exposure
            estimates.append(exposure_mean_contrast(outcome, exposure, exposure_probability, first=1, second=0)["contrast"])
        estimates = np.asarray(estimates)
        simulation_rows.append({"Degree": degree, "Assignment probability": probability, "Known effect": truth, "Mean estimate": estimates.mean(), "Standard deviation": estimates.std(ddof=1), "Root mean squared error": np.sqrt(np.mean((estimates - truth) ** 2)), "Largest inverse probability": 1 / exposure_probability.min()})
simulation_grid = pd.DataFrame(simulation_rows)
save_table(simulation_grid, "scope_simulation_grid", "Finite-sample variation across regular graph designs.", "tab:cim7-scope-simulation-grid", fit=True)
simulation_grid_book = simulation_grid[((simulation_grid["Degree"].isin([2, 12])) & (simulation_grid["Assignment probability"].isin([0.05, 0.25, 0.5])))][["Degree", "Assignment probability", "Mean estimate", "Standard deviation", "Root mean squared error", "Largest inverse probability"]].reset_index(drop=True)
save_table(simulation_grid_book, "scope_simulation_grid_book", "Selected finite-sample results across regular graph designs.", "tab:cim7-scope-simulation-grid-book", fit=True)

rule_rows = []
rule_graph = ring_graph(40, 2)
rule_degree = int(rule_graph[0].sum())
rule_baseline = 0.5 + 0.2 * np.sin(np.arange(40) / 3)
rule_effects = 0.6 + 0.1 * np.cos(np.arange(40) / 5)
rule_truth = float(rule_effects.mean())
for threshold in [1, 2]:
    for probability in [0.05, 0.10, 0.25, 0.50]:
        exposure_probability = exposure_probabilities(rule_graph, probability, threshold)
        rng = np.random.default_rng(simulation_random_state + 1000 * threshold + int(probability * 100))
        estimates = []
        for repetition in range(simulation_repetitions):
            assignment = rng.binomial(1, probability, 40)
            exposure = exposure_labels(rule_graph, assignment, threshold)
            outcome = rule_baseline + rule_effects * exposure
            estimates.append(exposure_mean_contrast(outcome, exposure, exposure_probability, first=1, second=0)["contrast"])
        estimates = np.asarray(estimates)
        rule_rows.append({"Required assigned neighbors": threshold, "Assignment probability": probability, "Probability of exposure one": exposure_probability[0, 1], "Largest inverse probability": 1 / exposure_probability.min(), "Mean estimate": estimates.mean(), "Standard deviation": estimates.std(ddof=1), "Root mean squared error": np.sqrt(np.mean((estimates - rule_truth) ** 2))})
rule_comparison = pd.DataFrame(rule_rows)
save_table(rule_comparison, "scope_rule_comparison", "Comparison of one-neighbor and two-neighbor exposure rules.", "tab:cim7-scope-rule-comparison", fit=True)

simulation_grid.to_csv(FIGURE_DATA / "scope_rmse_grid.csv", index=False)
figure, axis = plt.subplots(figsize=(8.8, 5.4), constrained_layout=True)
for degree, group in simulation_grid.groupby("Degree"):
    axis.plot(group["Assignment probability"], group["Root mean squared error"], marker="o", linewidth=2, label=f"Degree {degree}")
axis.set_xlabel("Assignment probability")
axis.set_ylabel("Root mean squared error")
axis.set_title("Finite-Sample Variation Across Regular Graphs")
axis.legend()
finish_axis(axis)
save_figure(figure, "scope_rmse_grid")

rule_comparison.to_csv(FIGURE_DATA / "scope_rule_comparison.csv", index=False)
figure, axis = plt.subplots(figsize=(8.8, 5.4), constrained_layout=True)
for threshold, group in rule_comparison.groupby("Required assigned neighbors"):
    axis.plot(group["Assignment probability"], group["Standard deviation"], marker="o", linewidth=2, label=f"At least {threshold}")
axis.set_xlabel("Assignment probability")
axis.set_ylabel("Estimator standard deviation")
axis.set_title("Changing the Exposure Rule Changes Finite-Sample Variation")
axis.legend()
finish_axis(axis)
save_figure(figure, "scope_rule_comparison")

,Degree,Assignment probability,Known effect,Mean estimate,Standard deviation,Root mean squared error,Largest inverse probability
0,2,0.05,0.613758,0.643934,0.814001,0.814398,10.25641
1,2,0.1,0.613758,0.617405,0.572033,0.571931,5.263158
2,2,0.25,0.613758,0.610429,0.371618,0.371558,2.285714
3,2,0.5,0.613758,0.61874,0.30894,0.308919,4
4,4,0.05,0.613758,0.601523,0.814891,0.81482,5.391017
5,4,0.1,0.613758,0.631563,0.573293,0.573455,2.907822
6,4,0.25,0.613758,0.622012,0.429792,0.429785,3.160494
7,4,0.5,0.613758,0.626487,0.511052,0.511109,16
8,8,0.05,0.613758,0.596294,0.82754,0.827559,2.971066
9,8,0.1,0.613758,0.630857,0.633742,0.633846,2.323057


,Degree,Assignment probability,Mean estimate,Standard deviation,Root mean squared error,Largest inverse probability
0,2,0.05,0.643934,0.814001,0.814398,10.25641
1,2,0.25,0.610429,0.371618,0.371558,2.285714
2,2,0.5,0.61874,0.30894,0.308919,4
3,12,0.05,0.636694,0.857472,0.857608,2.175616
4,12,0.25,0.569734,1.180383,1.180968,31.569292
5,12,0.5,0.745398,5.358112,5.358658,"4,096"


,Required assigned neighbors,Assignment probability,Probability of exposure one,Largest inverse probability,Mean estimate,Standard deviation,Root mean squared error
0,1,0.05,0.185494,5.391017,0.606812,0.79642,0.796291
1,1,0.1,0.3439,2.907822,0.614412,0.581027,0.580911
2,1,0.25,0.683594,3.160494,0.610089,0.417196,0.417128
3,1,0.5,0.9375,16,0.605631,0.526093,0.526051
4,2,0.05,0.014019,71.333036,0.613112,2.220182,2.219739
5,2,0.1,0.0523,19.120459,0.61944,1.224774,1.224543
6,2,0.25,0.261719,3.820896,0.622695,0.616043,0.615985
7,2,0.5,0.6875,3.2,0.605539,0.427562,0.427556


<Figure size 880x540 with 1 Axes>

<Figure size 880x540 with 1 Axes>

## Separating network description from causal scope

Let $m$ be the number of undirected edges among $N$ nodes. Graph density is

$$
\rho=\frac{2m}{N(N-1)}.
$$

The quantity $\rho$ describes how many possible links are present. It does not state why the links exist or whether a shock travels along them. The public correlation graph is used only to compare fixed topologies under generated assignments and outcomes.

In [3]:
public_returns = read_public_returns()
selected_returns = public_returns.iloc[:, :12]
correlation = selected_returns.corr()
industries = selected_returns.columns.to_list()
density_rows = []
for neighbor_count in range(1, 6):
    graph = correlation_graph(correlation, neighbor_count)
    degree = graph.sum(axis=1)
    probability = 0.25
    exposure_probability = exposure_probabilities(graph, probability)
    baseline = 0.4 + 0.08 * np.arange(len(industries))
    effects = np.full(len(industries), 0.6)
    truth = float(effects.mean())
    rng = np.random.default_rng(7400 + neighbor_count)
    estimates = []
    for repetition in range(3000):
        assignment = rng.binomial(1, probability, len(industries))
        exposure = exposure_labels(graph, assignment)
        outcome = baseline + effects * exposure
        estimates.append(exposure_mean_contrast(outcome, exposure, exposure_probability, first=1, second=0)["contrast"])
    estimates = np.asarray(estimates)
    density_rows.append({
        "Selected neighbors per node": neighbor_count,
        "Undirected edges": int(graph.sum() / 2),
        "Density": graph.sum() / (len(industries) * (len(industries) - 1)),
        "Minimum degree": int(degree.min()),
        "Maximum degree": int(degree.max()),
        "Smallest exposure probability": exposure_probability.min(),
        "Largest inverse probability": 1 / exposure_probability.min(),
        "Mean estimate": estimates.mean(),
        "Standard deviation": estimates.std(ddof=1),
        "Error from truth": estimates.mean() - truth,
    })
density_sweep = pd.DataFrame(density_rows)
save_table(density_sweep, "scope_public_density_sweep", "Generated exposure calculations across descriptive public topologies.", "tab:cim7-scope-public-density-sweep", fit=True)
density_sweep_book = density_sweep[["Selected neighbors per node", "Undirected edges", "Density", "Smallest exposure probability", "Largest inverse probability", "Standard deviation"]].copy()
save_table(density_sweep_book, "scope_public_density_sweep_book", "Readable density and exposure-rarity comparison.", "tab:cim7-scope-public-density-sweep-book", fit=True)

financial_context = pd.DataFrame([
    {"Statement": "Denser links can absorb sufficiently small shocks in the cited model", "Status": "Model specific"},
    {"Statement": "Denser links can propagate sufficiently large shocks in the cited model", "Status": "Model specific"},
    {"Statement": "The public return graph proves either mechanism", "Status": "Not supported"},
    {"Statement": "A correlation edge identifies a causal spillover", "Status": "Not supported"},
])
save_table(financial_context, "scope_financial_network_context", "Permitted interpretation of the financial network comparison.", "tab:cim7-scope-financial-network-context", fit=True)

claim_boundary = pd.DataFrame([
    {"Question": "Is the exposure mapping learned from returns?", "Answer": "No, it is supplied"},
    {"Question": "Are assignments observed in the archive?", "Answer": "No, they are generated"},
    {"Question": "Are potential outcomes observed in the archive?", "Answer": "No, they are generated"},
    {"Question": "Does an edge establish a spillover?", "Answer": "No"},
    {"Question": "Is a variance or confidence interval returned?", "Answer": "No"},
])
save_table(claim_boundary, "scope_claim_boundary", "Scientific boundaries for the network calculations.", "tab:cim7-scope-claim-boundary", fit=True)

section_summary = pd.DataFrame([
    {"Result": "Degree and assignment probability", "Value": "Determine marginal exposure probabilities"},
    {"Result": "Rare exposure", "Value": "Creates large inverse-probability contributions"},
    {"Result": "Exposure threshold", "Value": "Changes both the estimand and finite-sample variation"},
    {"Result": "Public graph", "Value": "Supplies topology only"},
    {"Result": "Financial shock comparison", "Value": "Retained only as a cited model-specific result"},
])
save_table(section_summary, "scope_section_summary", "Summary of density, positivity, and scope results.", "tab:cim7-scope-section-summary", fit=True)

density_sweep.to_csv(FIGURE_DATA / "scope_public_density_sweep.csv", index=False)
figure, left = plt.subplots(figsize=(8.8, 5.4), constrained_layout=True)
right = left.twinx()
left.plot(density_sweep["Density"], density_sweep["Standard deviation"], color=PALETTE["cyan"], marker="o", linewidth=2.2, label="Estimator standard deviation")
right.plot(density_sweep["Density"], density_sweep["Largest inverse probability"], color=PALETTE["amber"], marker="s", linewidth=2.2, label="Largest inverse probability")
left.set_xlabel("Descriptive graph density")
left.set_ylabel("Estimator standard deviation", color=PALETTE["cyan"])
right.set_ylabel("Largest inverse probability", color=PALETTE["amber"])
left.set_title("Density Changes Exposure Rarity Under the Supplied Rule")
left.grid(True, alpha=0.65)
lines = left.get_lines() + right.get_lines()
left.legend(lines, [line.get_label() for line in lines], loc="upper left")
save_figure(figure, "scope_public_density_sweep")

,Selected neighbors per node,Undirected edges,Density,Minimum degree,Maximum degree,Smallest exposure probability,Largest inverse probability,Mean estimate,Standard deviation,Error from truth
0,1,9,0.136364,1,3,0.25,4,0.597289,1.037829,-0.002711
1,2,16,0.242424,2,4,0.316406,3.160494,0.594962,1.070519,-0.005038
2,3,24,0.363636,3,6,0.177979,5.618656,0.604246,1.256168,0.004246
3,4,29,0.439394,4,7,0.133484,7.491541,0.601824,1.360936,0.001824
4,5,35,0.530303,5,10,0.056314,17.757727,0.64123,1.54383,0.04123


,Selected neighbors per node,Undirected edges,Density,Smallest exposure probability,Largest inverse probability,Standard deviation
0,1,9,0.136364,0.25,4,1.037829
1,2,16,0.242424,0.316406,3.160494,1.070519
2,3,24,0.363636,0.177979,5.618656,1.256168
3,4,29,0.439394,0.133484,7.491541,1.360936
4,5,35,0.530303,0.056314,17.757727,1.54383


,Statement,Status
0,Denser links can absorb sufficiently small sho...,Model specific
1,Denser links can propagate sufficiently large ...,Model specific
2,The public return graph proves either mechanism,Not supported
3,A correlation edge identifies a causal spillover,Not supported


,Question,Answer
0,Is the exposure mapping learned from returns?,"No, it is supplied"
1,Are assignments observed in the archive?,"No, they are generated"
2,Are potential outcomes observed in the archive?,"No, they are generated"
3,Does an edge establish a spillover?,No
4,Is a variance or confidence interval returned?,No


,Result,Value
0,Degree and assignment probability,Determine marginal exposure probabilities
1,Rare exposure,Creates large inverse-probability contributions
2,Exposure threshold,Changes both the estimand and finite-sample va...
3,Public graph,Supplies topology only
4,Financial shock comparison,Retained only as a cited model-specific result


<Figure size 880x540 with 2 Axes>